# Evaluation: 1 

In [ ]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from typing import List, Tuple
from CPL import query_preferences, load_graph
from transformers import BertTokenizer, BertModel
from sklearn.metrics import precision_score, recall_score, f1_score

# Load BERT tokenizer and model for graph building
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
bert_model.eval()

# Download stopwords once
nltk.download("stopwords", quiet=True)
STOPWORDS = set(stopwords.words("english"))
KEEP_WORDS = {"prefer", "dislike", "like", "love", "hate", "enjoy", "avoid"}
STOPWORDS = STOPWORDS - KEEP_WORDS

def preprocess(text: str) -> List[str]:
    """Lowercase, clean, tokenize, remove stopwords (but keep important preference words)."""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = text.split()
    return [t for t in tokens if t not in STOPWORDS]

def keyword_alignment(preferences: List[str], query: str, threshold: float = 0.3
                     ) -> Tuple[List[Tuple[str, float, List[str]]], List[Tuple[str, float, List[str]]]]:
    """Compute alignment between query and preferences."""
    query_kw = set(preprocess(query))
    if not query_kw:
        return [], []
    
    results = []
    for pref in preferences:
        pref_kw = set(preprocess(pref))
        if not pref_kw:
            continue
        common = query_kw.intersection(pref_kw)
        score = len(common) / min(len(query_kw), len(pref_kw))
        if score >= threshold:
            results.append((pref, score, list(common)))
    
    if not results:
        return [], []
    
    max_score = max(r[1] for r in results)
    best = [r for r in results if r[1] == max_score]
    
    return results, best

# ===========================
# MAIN PIPELINE
# ===========================
def evaluate_preferences(csv_path: str, output_path: str, bert_tokenizer, bert_model,
                         genre_filter=None, preference_type_filter=None, top_k=5, threshold=0.3):
    # Load dataset
    df = pd.read_csv(csv_path)
    df['userID'] = 'U001'

    results = []
    user_graphs = {}
    total = 0
    correct = 0

    # For metrics
    y_true = []
    y_pred = []
    coverage_hits = 0
    all_scores = []

    for _, row in df.iterrows():
        user_id = row["userID"]
        query_text = row["question_prompt"]
        gold_answer = row["answer"]
        if user_id not in user_graphs:
            user_graphs[user_id] = load_graph(user_id, './graphs')
            print(f"loaded graph for user {user_id}")

        # --- Get preferences from the graph ---
        graph_query = query_preferences(
            user_graphs[user_id], user_id, query_text,
            tokenizer=bert_tokenizer,
            model=bert_model,
            genre_filter=genre_filter,
            preference_type_filter=preference_type_filter,
            top_k=top_k,
            active_only=True
        )

        preferences = [entry['statement'] for entry in graph_query] if graph_query else []

        # --- Run keyword alignment ---
        aligned_prefs, best_prefs = keyword_alignment(preferences, query_text, threshold=threshold)

        # Collect scores for distribution
        for _, score, _ in aligned_prefs:
            all_scores.append(score)

        # --- Accuracy calculation (same as before) ---
        if preferences:
            total += 1
            if aligned_prefs:
                correct += 1

        # --- Precision/Recall/F1 setup ---
        # Binary: Did we align something (1) or not (0)?
        y_true.append(1 if gold_answer else 0)
        y_pred.append(1 if aligned_prefs else 0)

        # --- Coverage@k ---
        if preferences and gold_answer:
            # If gold answer keyword appears in any top-k pref
            if any(gold_answer.lower() in pref.lower() for pref in preferences):
                coverage_hits += 1

        results.append({
            "userID": user_id,
            "query_prompt": query_text,
            "gold_answer": gold_answer,
            "top_k_prefs": preferences,
            "aligned_prefs": aligned_prefs,
            "best_pref": best_prefs
        })

    # Save results as DataFrame
    out_df = pd.DataFrame(results)
    out_df.to_csv(output_path, index=False)
    print(f"✅ Results saved to {output_path}")

    # Accuracy
    if total > 0:
        accuracy = correct / total
        print(f"Accuracy: {accuracy:.3f} ({correct}/{total})")
    else:
        print("No preferences found for any user, accuracy not computed.")
        return

    # Precision / Recall / F1
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1 Score:  {f1:.3f}")

    # Coverage@k
    coverage = coverage_hits / total if total > 0 else 0.0
    print(f"Coverage@{top_k}: {coverage:.3f} ({coverage_hits}/{total})")

    # Alignment score distribution
    if all_scores:
        print(f"Alignment Score -> Mean: {pd.Series(all_scores).mean():.3f}, "
              f"Std: {pd.Series(all_scores).std():.3f}, "
              f"Min: {min(all_scores):.3f}, Max: {max(all_scores):.3f}")
    else:
        print("No alignment scores recorded.")

# Example usage
# evaluate_preferences(
#     csv_path="preference_cleaned.csv",
#     output_path="evaluation_results.csv",
#     bert_tokenizer=bert_tokenizer,
#     bert_model=bert_model
# )


In [ ]:
evaluate_preferences('./data/preference_cleaned.csv', './data/evaluations/evaluation_1_results.csv', bert_tokenizer, bert_model)

# Pinecone Evaluation

In [ ]:
# per_user_pinecone_eval.py
import os
import re
import time
import uuid
import numpy as np
import pandas as pd
import torch
import nltk
from tqdm import tqdm
from dotenv import load_dotenv
from typing import List, Tuple
from transformers import BertTokenizer, BertModel
from sklearn.metrics import precision_score, recall_score, f1_score

# Pinecone: using the same import style you provided
from pinecone import Pinecone

# -----------------------
# Config + Init
# -----------------------
load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
if not PINECONE_API_KEY:
    raise RuntimeError("PINECONE_API_KEY not found in environment")

INDEX_NAME = "preference-prompts"
MODEL_NAME = "bert-base-uncased"
BATCH_UPSERT_SIZE = 100  # upsert in batches for speed
TOP_K = 5
THRESHOLD = 0.3

os.makedirs("data/evaluations/graph_evaluations", exist_ok=True)
os.makedirs("data/pinecone_matches", exist_ok=True)

# -----------------------
# Load model/tokenizer
# (same as your storing code)
# -----------------------
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertModel.from_pretrained(MODEL_NAME)
model.eval()

# -----------------------
# NLTK stopwords for evaluation (keyword alignment)
# -----------------------
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words("english"))
KEEP_WORDS = {"prefer", "dislike", "like", "love", "hate", "enjoy", "avoid"}
STOPWORDS = STOPWORDS - KEEP_WORDS

# -----------------------
# Utility: get CLS embedding
# -----------------------
def get_embedding(text: str):
    tokens = tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding="max_length")
    with torch.no_grad():
        outputs = model(**tokens)
    cls_embedding = outputs.last_hidden_state[:, 0, :]  # CLS token
    return cls_embedding.squeeze().numpy()

# -----------------------
# Helper: robustly get vector count from index
# -----------------------
def get_index_vector_count(index) -> int:
    try:
        stats = index.describe_index_stats()
        # try common shapes
        if isinstance(stats, dict):
            if "total_vector_count" in stats:
                return int(stats["total_vector_count"])
            namespaces = stats.get("namespaces", {})
            if namespaces:
                return int(sum(ns.get("vector_count", 0) for ns in namespaces.values()))
        # fallback: some older clients may return object-like - try attributes
        if hasattr(stats, "total_vector_count"):
            return int(getattr(stats, "total_vector_count"))
    except Exception as e:
        print("Warning: describe_index_stats failed:", e)
    # unknown -> return -1 to indicate unknown
    return -1

# -----------------------
# Keyword alignment functions (your earlier evaluation code)
# -----------------------
def preprocess(text: str) -> List[str]:
    """Lowercase, clean, tokenize, remove stopwords (but keep important preference words)."""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = text.split()
    return [t for t in tokens if t not in STOPWORDS]

def keyword_alignment(preferences: List[str], query: str, threshold: float = 0.3
                     ) -> Tuple[List[Tuple[str, float, List[str]]], List[Tuple[str, float, List[str]]]]:
    """Compute alignment between query and preferences."""
    query_kw = set(preprocess(query))
    if not query_kw:
        return [], []
    
    results = []
    for pref in preferences:
        pref_kw = set(preprocess(pref))
        if not pref_kw:
            continue
        common = query_kw.intersection(pref_kw)
        score = len(common) / min(len(query_kw), len(pref_kw))
        if score >= threshold:
            results.append((pref, score, list(common)))
    
    if not results:
        return [], []
    
    max_score = max(r[1] for r in results)
    best = [r for r in results if r[1] == max_score]
    
    return results, best

# -----------------------
# Evaluation routine (adapted from your provided evaluator)
# -----------------------
def evaluate_pinecone_results(csv_path: str, output_path: str, threshold: float = 0.3, top_k: int = 5):
    df = pd.read_csv(csv_path)
    # keep userID present (if missing default)
    if "userID" not in df.columns:
        df["userID"] = df.get("userID", "U001")

    results = []
    total = 0
    correct = 0

    y_true = []
    y_pred = []
    coverage_hits = 0
    all_scores = []

    for _, row in df.iterrows():

        user_id = row["userID"]
        query_text = row.get("question_prompt", "")
        gold_answer = row.get("answer", "")

        # statements column may be list-like stored as string
        statements = row.get("statements", [])
        if isinstance(statements, str):
            try:
                statements = eval(statements)
                if not isinstance(statements, list):
                    statements = [statements]
            except Exception:
                statements = [statements]

        preferences = statements[:top_k]

        aligned_prefs, best_prefs = keyword_alignment(preferences, query_text, threshold=threshold)

        for _, score, _ in aligned_prefs:
            all_scores.append(score)

        if preferences:
            total += 1
            if aligned_prefs:
                correct += 1

        y_true.append(1 if gold_answer else 0)
        y_pred.append(1 if aligned_prefs else 0)

        if preferences and gold_answer:
            if any(str(gold_answer).lower() in str(pref).lower() for pref in preferences):
                coverage_hits += 1

        results.append({
            "userID": user_id,
            "query_prompt": query_text,
            "gold_answer": gold_answer,
            "top_k_prefs": preferences,
            "aligned_prefs": aligned_prefs,
            "best_pref": best_prefs,
        })

    out_df = pd.DataFrame(results)
    out_df.to_csv(output_path, index=False)
    print(f"✅ Evaluation results saved to {output_path}")

    # Metrics
    if total > 0:
        accuracy = correct / total
        print(f"\nAccuracy: {accuracy:.3f} ({correct}/{total})")
    else:
        print("No preferences found, skipping accuracy.")

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1 Score:  {f1:.3f}")

    coverage = coverage_hits / total if total > 0 else 0.0
    print(f"Coverage@{top_k}: {coverage:.3f} ({coverage_hits}/{total})")

    if all_scores:
        s = pd.Series(all_scores)
        print(f"Alignment Score -> Mean: {s.mean():.3f}, Std: {s.std():.3f}, Min: {s.min():.3f}, Max: {s.max():.3f}")
    else:
        print("No alignment scores recorded.")

# -----------------------
# Main loop: flush, upsert one user's prefs, query all questions, evaluate
# -----------------------
def main():
    # Connect to Pinecone
    pc = Pinecone(api_key=PINECONE_API_KEY)
    index = pc.Index(INDEX_NAME)

    # Load dataset
    df = pd.read_csv("./data/preference_cleaned.csv")
    if "userID" not in df.columns:
        raise RuntimeError("Dataset is missing 'userID' column")

    unique_users = sorted(df["userID"].unique())
    print("Will process users:", unique_users)

    for user in unique_users:
        print("\n" + "="*60)
        print(f"Processing user: {user}")

        # --- Flush index ---
        try:
            index.delete(delete_all=True)
            time.sleep(1.0)
        except Exception as e:
            print("Warning: flush failed:", e)

        print(f"After flush, VDB vector count: {get_index_vector_count(index)}")

        # --- Upsert only this user's preferences ---
        user_rows = df[df["userID"] == user]
        if user_rows.empty:
            print(f"No rows found for user {user}, skipping upsert and evaluation.")
            continue

        batch, upserted = [], 0
        for _, row in tqdm(user_rows.iterrows(), total=len(user_rows), desc=f"Upserting user {user}"):
            pref = row["preference_prompt"]
            emb = get_embedding(pref)
            vector_id = f"{row['userID']}-{uuid.uuid4()}"
            md = {"preference_prompt": pref, "userID": row["userID"]}
            batch.append((vector_id, emb.tolist(), md))
            if len(batch) >= BATCH_UPSERT_SIZE:
                index.upsert(batch)
                upserted += len(batch)
                batch = []
        if batch:
            index.upsert(batch)
            upserted += len(batch)

        time.sleep(1.0)
        print(f"After upsert (user {user}), VDB vector count: {get_index_vector_count(index)} (upserted {upserted} vectors)")

                # --- Query Pinecone ---
        output_rows = []
        latencies = []  # collect all query latencies for this user
        print(f"\nQuerying VDB for {user}'s question prompts and collecting top matches...")
        for _, qrow in tqdm(user_rows.iterrows(), total=len(user_rows), desc=f"Querying questions for {user}"):
            question_prompt = qrow["question_prompt"]
            query_embedding = get_embedding(question_prompt)

            try:
                start = time.time()
                query_result = index.query(vector=query_embedding.tolist(), top_k=TOP_K, include_metadata=True)
                latency = time.time() - start
                matches = getattr(query_result, "matches", None) or query_result.get("matches", [])
            except Exception as e:
                print("Query failed:", e)
                matches, latency = [], 0

            latencies.append(latency)  # store latency

            matched_statements = []
            for match in matches:
                meta = getattr(match, "metadata", None) or match.get("metadata", {})
                matched_statements.append(meta.get("preference_prompt", meta))

            output_rows.append({
                "userID": qrow["userID"],
                "preference_prompt": qrow["preference_prompt"],
                "question_prompt": question_prompt,
                "answer": qrow["answer"],
                "statements": matched_statements,
                "query_latency_sec": latency,
            })

        # --- Compute latency stats ---
        if latencies:
            avg_latency = np.mean(latencies)
            p95_latency = np.percentile(latencies, 95)
            print(f"\nLatency Stats for user {user}: "
                  f"Avg = {avg_latency:.4f}s, P95 = {p95_latency:.4f}s")
        else:
            print("No latencies recorded.")
            avg_latency = p95_latency = 0.0

        # --- Evaluate directly without storing matches CSV ---
        matches_df = pd.DataFrame(output_rows)
        eval_out = f"data/evaluations/graph_evaluations/pinecone_eval_results_user_{user}.csv"
        print(f"\nRunning evaluation for user {user} (results -> {eval_out})")

        # Save matches_df temporarily inside evaluate_pinecone_results
        tmp_path = f"/matches_user_{user}.csv"
        matches_df.to_csv(tmp_path, index=False)
        evaluate_pinecone_results(csv_path=tmp_path, output_path=eval_out, threshold=THRESHOLD, top_k=TOP_K)

        # Save latency stats alongside evaluation results
        latency_out = {
            "userID": user,
            "Avg_latency_sec": avg_latency,
            "P95_latency_sec": p95_latency
        }
        latency_df = pd.DataFrame([latency_out])
        latency_csv = f"data/evaluations/graph_evaluations/pinecone_latency_user_{user}.csv"
        latency_df.to_csv(latency_csv, index=False)
        print(f"✅ Latency results saved to {latency_csv}")

        print(f"Completed evaluation for user {user}")

    print("\nAll done for users:", unique_users)


if __name__ == "__main__":
    main()


## Evaluation Results

### Multiple Users (5 Graphs)

| Metric                        | Result                          | Notes |
|-------------------------------|---------------------------------|-------|
| Accuracy                      | **0.874 (348/398)**             | Correct alignment rate |
| Recall                        | 0.874                           | High coverage of relevant prefs |
| F1 Score                      | 0.933                           | Strong balance of precision/recall |
| Coverage@5                    | 0.701 (279/398)                 | Top-5 prefs contain correct one ~70% |
| Alignment Score (Mean ± Std)  | 0.462 ± 0.167 (Min: 0.333, Max: 1.0) | Stable alignment distribution |
| Accuracy_before               | 0.889                           | Before adding new prefs |
| Accuracy_after                | 0.889                           | After adding new prefs |
| Forgetting Rate               | **0.0**                         | No catastrophic forgetting |
| Avg Latency / Query           | 0.085 sec                       | Very efficient on CPU |
| P95 Latency                   | 0.103 sec                       | 95% queries < 0.1s |

---

### Combined Users (1 Graph with 400 prefs)

| Metric                        | Result                          | Notes |
|-------------------------------|---------------------------------|-------|
| Accuracy                      | 0.814                           | Slightly lower when prefs combined |
| Recall                        | 0.824                           | Drop due to interference/noise |
| F1 Score                      | 0.904                           | Still strong balance |
| Coverage@5                    | 0.595 (237/398)                 | Only ~60% success in top-5 |
| Alignment Score (Mean ± Std)  | 0.462 ± 0.154 (Min: 0.333, Max: 1.0) | Distribution stable |
| Accuracy_before               | 0.814                           | Before new prefs |
| Accuracy_after                | 0.814                           | After new prefs |
| Forgetting Rate               | **0.0**                         | No forgetting (graph-based retention) |
| Avg Latency / Query           | 0.104 sec                       | Still fast despite more prefs |
| P95 Latency                   | 0.171 sec                       | Worst case under 0.2s |

### PINECONE (VDB)
#### combined user
| Metric                    | Value           |
|---------------------------|------------------|
| **Accuracy**              | 0.643 (256/398)  |
| **Precision**             | 1.000            |
| **Recall**                | 0.643            |
| **F1 Score**              | 0.783            |
| **Coverage@5**            | 0.131 (52/398)   |
| **Alignment Mean Score**  | 0.373            |
| **Alignment Std Dev**     | 0.098            |
| **Alignment Min Score**   | 0.333            |
| **Alignment Max Score**   | 0.800            |

##### ⏱ Latency Metrics

| Metric                        | Value           |
|-------------------------------|------------------|
| **Total Evaluation Time**     | 0.002 seconds    |
| **Average Latency/Query**     | 0.000 seconds    |
| **P95 Latency**               | 0.000 seconds    |


#### Multi User
| Metric                | Calculation                                                                 | Mean   |
|------------------------|-----------------------------------------------------------------------------|--------|
| **Accuracy**           | (0.375 + 0.838 + 0.944 + 0.914 + 0.925) / 5                                | 0.799  |
| **Precision**          | (1 + 1 + 1 + 1 + 1) / 5                                                    | 1.000  |
| **Recall**             | (0.338 + 0.775 + 0.838 + 0.810 + 0.785) / 5                                | 0.709  |
| **F1 Score**           | (0.505 + 0.873 + 0.912 + 0.895 + 0.879) / 5                                | 0.813  |
| **Coverage@5**         | (0.250 + 0.365 + 0.451 + 0.286 + 0.328) / 5                                | 0.336  |
| **Alignment Score**    | (0.559 + 0.440 + 0.394 + 0.385 + 0.411) / 5                                | 0.438  |



# Evaluation 1.1 (forgetting and latency)

In [ ]:
import re
import time
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from typing import List, Tuple
from CPL import query_preferences, load_graph
from transformers import BertTokenizer, BertModel
from sklearn.metrics import precision_score, recall_score, f1_score

# ======================
# Setup
# ======================
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
bert_model.eval()

nltk.download("stopwords", quiet=True)
STOPWORDS = set(stopwords.words("english"))
KEEP_WORDS = {"prefer", "dislike", "like", "love", "hate", "enjoy", "avoid"}
STOPWORDS = STOPWORDS - KEEP_WORDS

def preprocess(text: str) -> List[str]:
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = text.split()
    return [t for t in tokens if t not in STOPWORDS]

def keyword_alignment(preferences: List[str], query: str, threshold: float = 0.3
                     ) -> Tuple[List[Tuple[str, float, List[str]]], List[Tuple[str, float, List[str]]]]:
    query_kw = set(preprocess(query))
    if not query_kw:
        return [], []
    
    results = []
    for pref in preferences:
        pref_kw = set(preprocess(pref))
        if not pref_kw:
            continue
        common = query_kw.intersection(pref_kw)
        score = len(common) / min(len(query_kw), len(pref_kw))
        if score >= threshold:
            results.append((pref, score, list(common)))
    
    if not results:
        return [], []
    
    max_score = max(r[1] for r in results)
    best = [r for r in results if r[1] == max_score]
    return results, best


# ======================
# Evaluation
# ======================
def evaluate_forgetting_latency(csv_path: str, output_path: str,
                                bert_tokenizer, bert_model,
                                genre_filter=None, preference_type_filter=None,
                                top_k=5, threshold=0.3):

    df = pd.read_csv(csv_path)
    # df = df.sort_values("userID").reset_index(drop=True)
    df['UserID'] = 'U001'

    # Split dataset (first half = old prefs, second half = new prefs)
    midpoint = len(df) // 2
    old_df = df.iloc[:midpoint]
    new_df = df.iloc[midpoint:]

    user_graphs = {}
    results = []

    # ---------------------------
    # Step A: Accuracy on OLD prefs only
    # ---------------------------
    total_before, correct_before = 0, 0
    y_true_before, y_pred_before = [], []

    for _, row in old_df.iterrows():
        user_id, query_text, gold_answer = row["userID"], row["question_prompt"], row["answer"]

        if user_id not in user_graphs:
            user_graphs[user_id] = load_graph(user_id, './graphs')

        graph_query = query_preferences(
            user_graphs[user_id], user_id, query_text,
            tokenizer=bert_tokenizer, model=bert_model,
            genre_filter=genre_filter, preference_type_filter=preference_type_filter,
            top_k=top_k, active_only=True
        )
        prefs = [entry['statement'] for entry in graph_query] if graph_query else []
        aligned_prefs, best_prefs = keyword_alignment(prefs, query_text, threshold)

        total_before += 1
        if aligned_prefs:
            correct_before += 1
            y_pred_before.append(1)
        else:
            y_pred_before.append(0)
        y_true_before.append(1)  # since each query has a gold preference

    acc_before = correct_before / total_before if total_before > 0 else 0.0

    # ---------------------------
    # Step B: Add NEW prefs + recheck OLD queries
    # ---------------------------
    total_after, correct_after = 0, 0
    y_true_after, y_pred_after = [], []

    for _, row in new_df.iterrows():
        user_id = row["userID"]
        if user_id not in user_graphs:
            user_graphs[user_id] = load_graph(user_id, './graphs')

    for _, row in old_df.iterrows():
        user_id, query_text, gold_answer = row["userID"], row["question_prompt"], row["answer"]

        graph_query = query_preferences(
            user_graphs[user_id], user_id, query_text,
            tokenizer=bert_tokenizer, model=bert_model,
            genre_filter=genre_filter, preference_type_filter=preference_type_filter,
            top_k=top_k, active_only=True
        )
        prefs = [entry['statement'] for entry in graph_query] if graph_query else []
        aligned_prefs, best_prefs = keyword_alignment(prefs, query_text, threshold)

        total_after += 1
        if aligned_prefs:
            correct_after += 1
            y_pred_after.append(1)
        else:
            y_pred_after.append(0)
        y_true_after.append(1)

    acc_after = correct_after / total_after if total_after > 0 else 0.0

    # Forgetting Rate
    forgetting_rate = (acc_before - acc_after) / acc_before if acc_before > 0 else 0.0

    # ---------------------------
    # Step C: Latency Measurement
    # ---------------------------
    latencies = []
    for _, row in df.iterrows():
        user_id, query_text = row["userID"], row["question_prompt"]
        start = time.time()
        graph_query = query_preferences(
            user_graphs[user_id], user_id, query_text,
            tokenizer=bert_tokenizer, model=bert_model,
            genre_filter=genre_filter, preference_type_filter=preference_type_filter,
            top_k=top_k, active_only=True
        )
        latencies.append(time.time() - start)
        prefs = [entry['statement'] for entry in graph_query] if graph_query else []
        aligned_prefs, best_prefs = keyword_alignment(prefs, query_text, threshold)

    avg_latency = np.mean(latencies)
    p95_latency = np.percentile(latencies, 95)

    # ---------------------------
    # Store results
    # ---------------------------
    results_summary = {
        "Accuracy_before": acc_before,
        "Accuracy_after": acc_after,
        "Forgetting_rate": forgetting_rate,
        "Avg_latency_sec": avg_latency,
        "P95_latency_sec": p95_latency
    }

    results_df = pd.DataFrame([results_summary])
    results_df.to_csv(output_path, index=False)
    print(f"✅ Results saved to {output_path}")
    print(results_summary)



In [ ]:
if __name__ == "__main__":
    evaluate_forgetting_latency(
        csv_path="./data/preference_cleaned.csv",
        output_path="./data/evaluation_1_1_results_single_graph.csv",
        bert_tokenizer=bert_tokenizer,
        bert_model=bert_model
    )

# Evaluation 2

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from transformers import BertTokenizer, BertModel
from peft import LoraConfig, TaskType

import whisper
import pandas as pd
import torch
from dotenv import load_dotenv
import os

from sklearn.model_selection import KFold
import pandas as pd, os
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

from adapter_management import AdapterManager
from query_router import QueryRouter
from Implicit import Implicit_main
from langchain_groq import ChatGroq
from utils.LLM_judge import evaluate_response
from utils.federation import perform_federated_learning


In [ ]:
load_dotenv()
HF_TOKEN = os.getenv("HUGGING_FACE_ACCESS_TOKEN")
# Set device to CPU explicitly
device = torch.device("cpu")
torch.set_default_device(device)

# Initialize models
llama_model_name = "meta-llama/Llama-3.2-1B-Instruct"
llama_torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# Load the base model
model = AutoModelForCausalLM.from_pretrained(
    llama_model_name,
    token=HF_TOKEN,
    torch_dtype=llama_torch_dtype,
    device_map="cpu",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(
    llama_model_name,
    token=HF_TOKEN,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
LLM_MODEL = os.getenv("LLM_MODEL")
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name=LLM_MODEL,
)


# model.train()

# Initialize other models
whisper_model = whisper.load_model("base")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
sentiment_pipeline = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

# Load BERT tokenizer and model for graph building
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
bert_model.eval()

print("Models loaded successfully!")


In [ ]:
adapter_manager = AdapterManager(base_model_name=llama_model_name)
adapter_manager.load_metadata()

In [ ]:
def train_local_adapter_from_preferences(csv_file, adapter_manager, base_model, peft_config, embedding_model, sentiment_pipeline, whisper_model):
    """
    For each row in the CSV, create an input-output pair and train the local adapter using Implicit_main.
    The input-output pair format: "<|user|> [Original user query]\n<|assistant|> [Corrected assistant response]"
    """
    df = pd.read_csv(csv_file)
    print(f"Loaded {len(df)} examples from {csv_file}")

    for idx, row in df.iterrows():
        user_id = row['userID']
        response = row['preference_prompt']
        user_query = row.get('question_prompt', '')
        answer = row.get('answer','')

        # Skip if either field is missing
        if not isinstance(user_query, str) or not isinstance(response, str):
            continue

        # Create the input-output pair
        io_pair = f"""### Instruction:
                    {response}
                    
                    ### Input:
                    {user_query}
                    
                    ### Response:
                    {answer}"""

        # Get local and global models for the user
        local_model, _ = adapter_manager.initialize_user(user_id, base_model, peft_config)

        # Prepare input for Implicit_main (expects a list of prompts)
        input_data = [io_pair]

        # Call Implicit_main to train the local adapter
        Implicit_main(
            input_data=input_data,
            whisper_model=whisper_model,
            model=local_model,
            embedding_model=embedding_model,
            sentiment_pipeline=sentiment_pipeline,
            voice_metadata=None,
            user_id=user_id,
            adapter_manager=adapter_manager
        )

        print(f"Trained local adapter for user {user_id} on row {idx+1}/{len(df)}")

def query_huggingface_model(query_text, preference_statements, model, tokenizer):
    """
    Generate a response using a HuggingFace model with preference context
    
    Args:
        query_text: The query to answer
        preference_statements: Context from preferences (can be empty string)
        model: The HuggingFace model to use
        tokenizer: Tokenizer for the model
        
    Returns:
        Generated response
    """
    try:
        # Create the prompt with preferences if provided
        if preference_statements:
            if isinstance(preference_statements, list):
                preference_text = "\n".join(preference_statements)
            else:
                preference_text = preference_statements
                
            prompt = f"""<|system|>
                You are a helpful AI assistant that answers questions based on user preferences. 
                Use these preferences to guide your response:
                {preference_text}
                <|user|>
                {query_text}
                <|assistant|>"""
        else:
            prompt = f"""<|system|>
            You are a helpful AI assistant. Answer the provided query.
            <|user|>
            {query_text}
            <|assistant|>"""

        # Generate the response
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        # Set generation parameters
        generation_kwargs = {
            "max_new_tokens": 70,
            "do_sample": True,
            "temperature": 0.9,
            "repetition_penalty": 1.1,
            "pad_token_id": tokenizer.pad_token_id
        }
        
        # Generate response
        with torch.inference_mode():
            output = model.generate(**inputs, **generation_kwargs)
        
        # Decode and clean up the response
        response = tokenizer.decode(output[0], skip_special_tokens=True)
        
        # Extract the assistant's reply (everything after the last <|assistant|> tag)
        if "<|assistant|>" in response:
            response = response.split("<|assistant|>")[-1].strip()
        
        return response
            
    except Exception as e:
        print(f"Error in query_huggingface_model: {e}")
        return "I'm sorry, I encountered an error while trying to answer."

def process_row(idx, row, fl_type, global_model):
    user_id = row['userID']
    print(f"[Thread] User {user_id}")
    query_text = row['question_prompt']

    for i in range(3):
        response = query_huggingface_model(query_text, None, global_model, tokenizer)

        if response not in ['', None, ' ']:
            evaluation_output = evaluate_response(
                query_text,
                row['answer'],
                row['preference_prompt'],
                response,
                llm=llm
            )
            verdict = evaluation_output.get("verdict", "")
            feedback = evaluation_output.get("feedback", "")
            print(f"LLM response: {response}\nFeedback: {feedback}")

            with results_lock:
                results.append({
                    'query_text': query_text,
                    'response': response,
                    'is_correct': verdict,
                    'feedback': feedback,
                    'FL_type': fl_type
                })

            print(f"Example {idx+1}: {'✓' if verdict == 1 else '✗'} | Expected: {row['answer']}")
            break  # stop retrying if success

# Load and sort the data
df = pd.read_csv('/kaggle/input/pref-data/preference_cleaned.csv')
df = df.head(13)
df = df.sort_values(by="userID").reset_index(drop=True)
llm = llm

# Set up 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

folds = []
for fold, (train_idx, test_idx) in enumerate(kf.split(df)):
    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_test = df.iloc[test_idx].reset_index(drop=True)
    folds.append((df_train, df_test))
    print(f"Fold {fold+1}: Train size = {len(df_train)}, Test size = {len(df_test)}")
_adapter = ["flora","default","fedavg"]

for x in _adapter:
    for i in range(len(folds)):
        df_train_1, df_test_1 = folds[i]
        df_train_1.to_csv('./training_data.csv')
        train_local_adapter_from_preferences('./training_data.csv', adapter_manager, model, peft_config, embedding_model, sentiment_pipeline, whisper_model)
        global_model = perform_federated_learning(model, peft_config, adapter_manager, x)
        results_lock = threading.Lock()  # to safely update shared vars
        results = []
        first_save = True
        output_path = f"evaluation_results_fold{i}_{x}.csv"

        df = df_test_1

        # Run with 2 threads
        with ThreadPoolExecutor(max_workers=2) as executor:
            futures = [executor.submit(process_row, idx, row, x, global_model) for idx, row in df.iterrows()]
            for future in as_completed(futures):
                # batch saving inside loop
                with results_lock:
                    if len(results) >= 10:
                        results_df = pd.DataFrame(results)
                        write_mode = 'w' if first_save else 'a'
                        header = first_save
                        results_df.to_csv(output_path, index=False, mode=write_mode, header=header)
                        first_save = False
                        results.clear()
                        print(f"✅ Saved batch ({len(results_df)})")

        # Final leftover save
        if results:
            results_df = pd.DataFrame(results)
            results_df.to_csv(output_path, index=False, mode='a', header=not os.path.exists(output_path))

        # Compute accuracy from results DataFrame
        results_df = pd.read_csv(output_path)
        correct = (results_df['is_correct'] == 1).sum()
        total = len(results_df)
        accuracy = correct / total if total > 0 else 0
        print(f"\nFinal Accuracy: {accuracy:.2%} ({correct}/{total})")
        os.system("rm -rf /kaggle/working/models")
    
    

In [ ]:
import os
import pandas as pd
import numpy as np

# Directories containing results
methods = ["fedavg", "default", "vanilla","combined_adapters"]  # adjust if naming differs
n_folds = 5


results = {}
_directory = './data/evaluations'

for method in methods:
    accuracies = []
    for i in range(n_folds):
        file_path = os.path.join(_directory, method, f"combined_agree_fold{i}_{method}.csv")
        if method == "combined_adapters":
            file_path = os.path.join(_directory, method, f"combined_agree_fold{i}_{method}.csv")
        if not os.path.exists(file_path):
            print(f"⚠️ Missing file: {file_path}")
            continue
        
        # Load CSV
        df = pd.read_csv(file_path)
        
        # Expect a column "is_correct" with 0/1
        acc = df["llm1_score"].mean()
        accuracies.append(acc)
    
    if accuracies:
        mean_acc = np.mean(accuracies)
        std_acc = np.std(accuracies)
        results[method] = {
            "per_fold": accuracies,
            "mean": mean_acc,
            "std": std_acc
        }

# Print results
print("\n=== Accuracy Results ===")
for method, vals in results.items():
    print(f"{method.upper():<10} | Mean Acc: {vals['mean']:.4f} | Std: {vals['std']:.4f} | Per-fold: {vals['per_fold']}")


### === Accuracy Results ===

| Method              | Mean Acc | Std    | Per-fold                                                                                         |
|---------------------|---------:|-------:|:--------------------------------------------------------------------------------------------------|
| FEDAVG              | 0.4274   | 0.0564 | [0.453125, 0.362069, 0.482759, 0.357143, 0.482143]                                               |
|FEDDPA              | 0.5276   | 0.0715 | [0.4250, 0.6375, 0.5500, 0.5443, 0.4810] |
| DEFAULT             | 0.3969   | 0.0433 | [0.406780, 0.393443, 0.383333, 0.333333, 0.467742]                                               |
| VANILLA             | 0.3951   | 0.0617 | [0.355932, 0.328125, 0.393443, 0.388889, 0.509091]                                               |
| COMBINED_ADAPTERS   | 0.4089   | 0.0429 | [0.377358, 0.350000, 0.474576, 0.428571, 0.413793]                                               |

# Evaluation 3 (complete evaluation)

In [ ]:
import re
import pandas as pd
import numpy as np
from typing import List, Tuple
from sklearn.metrics import precision_score, recall_score, f1_score

# -----------------------------
# Text Preprocessing & Alignment
# -----------------------------
def preprocess(text: str) -> List[str]:
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = text.split()
    return tokens

def keyword_alignment(preferences: List[str], query: str, threshold: float = 0.3
                     ) -> Tuple[List[Tuple[str, float, List[str]]], List[Tuple[str, float, List[str]]]]:
    query_kw = set(preprocess(query))
    if not query_kw:
        return [], []
    
    results = []
    for pref in preferences:
        pref_kw = set(preprocess(pref))
        if not pref_kw:
            continue
        common = query_kw.intersection(pref_kw)
        score = len(common) / min(len(query_kw), len(pref_kw))
        if score >= threshold:
            results.append((pref, score, list(common)))
    
    if not results:
        return [], []
    
    max_score = max(r[1] for r in results)
    best = [r for r in results if r[1] == max_score]
    
    return results, best

# -----------------------------
# MAIN EVALUATION
# -----------------------------

def evaluate_pipeline(csv_path: str, threshold: float = 0.3, window: int = 5):
    df = pd.read_csv(csv_path, sep=",")  # adjust sep="," if CSV
    df['score'] = df['score'].astype(int)

    # ============ Metrics ============

    # 1. Overall Accuracy
    overall_acc = df['score'].mean()

    # 2. Accuracy per Adapter (kept for internal calc, but not printed in detail)
    adapter_acc = df.groupby('adapter_used')['score'].mean()

    # 3. Graph Utilization Rate (statements non-empty)
    df['graph_used'] = df['statements'].apply(lambda x: len(str(x).strip("[]")) > 0)
    graph_utilization = df['graph_used'].mean()

    # 4. Preference Alignment Check (using keyword overlap)
    alignment_hits = 0
    total_with_prefs = 0
    alignment_scores = []

    for _, row in df.iterrows():
        query = row['question_prompt']
        prefs = eval(row['statements']) if isinstance(row['statements'], str) else []
        if prefs:
            total_with_prefs += 1
            aligned, best = keyword_alignment(prefs, query, threshold=threshold)
            if aligned:
                alignment_hits += 1
                alignment_scores.extend([s for _, s, _ in aligned])

    alignment_acc = alignment_hits / total_with_prefs if total_with_prefs > 0 else 0.0
    alignment_stats = {
        "mean": np.mean(alignment_scores) if alignment_scores else 0.0,
        "std": np.std(alignment_scores) if alignment_scores else 0.0,
        "min": np.min(alignment_scores) if alignment_scores else 0.0,
        "max": np.max(alignment_scores) if alignment_scores else 0.0
    }

    # 5. Adapter Performance (same as #2, kept for calc but summarized in print)
    adapter_perf = adapter_acc

    # 6. Adapter Selection Accuracy (avg only)
    adapter_selection_acc = adapter_perf.mean()

    # 7. Retention Stability (rolling accuracy per user)
    retention_scores = {}
    for user, group in df.groupby('userID'):
        group = group.reset_index(drop=True)
        rolling_acc = group['score'].rolling(window=window, min_periods=1).mean()
        retention_scores[user] = rolling_acc.min() / rolling_acc.max() if rolling_acc.max() > 0 else 1.0
    avg_retention = np.mean(list(retention_scores.values()))

    # ============ New: Adapter Contributions Summary ============
    # Group by adapter to show contributions (queries handled, accuracy, and identify best)
    # Aggregate "blended" sub-variants into a single "blended" adapter
    df['adapter_used'] = df['adapter_used'].apply(lambda x: 'blended' if 'blended' in str(x).lower() else x)
    adapter_contributions = df.groupby('adapter_used').agg(
        queries_handled=('score', 'count'),  # Total queries per adapter
        correct_predictions=('score', 'sum'),  # Correct predictions
        accuracy=('score', 'mean')  # Accuracy per adapter
    ).reset_index()
    # Identify the "best" adapter (highest accuracy)
    if not adapter_contributions.empty:
        best_adapter = adapter_contributions.loc[adapter_contributions['accuracy'].idxmax()]
    else:
        best_adapter = None

    # ============ Print Results (Simplified) ============
    print("\n--- Evaluation Metrics ---")
    print(f"Overall Accuracy: {overall_acc * 100:.1f}%")
    print(f"Graph Utilization Rate: {graph_utilization * 100:.1f}%")
    print(f"Preference Alignment Accuracy: {alignment_acc * 100:.1f}%")
    print(f"Alignment Score Stats: Mean: {alignment_stats['mean'] * 100:.1f}%, Std: {alignment_stats['std'] * 100:.1f}%, Min: {alignment_stats['min'] * 100:.1f}%, Max: {alignment_stats['max'] * 100:.1f}%")
    print(f"Adapter Selection Accuracy (avg): {adapter_selection_acc * 100:.1f}%")
    print(f"Average Retention Stability: {avg_retention * 100:.1f}%")
    
    # New concise adapter summary (aggregated for blended)
    print("\n--- Adapter Contributions Summary ---")
    if best_adapter is not None:
        print(f"Best Adapter: {best_adapter['adapter_used']} (Accuracy: {best_adapter['accuracy'] * 100:.1f}%)")
        print("Adapter Breakdown (Queries Handled | Accuracy):")
        for _, row in adapter_contributions.iterrows():
            print(f"  - {row['adapter_used']}: {int(row['queries_handled'])} queries | {row['accuracy'] * 100:.1f}%")
    else:
        print("No adapter data available.")

    return {
        "overall_acc": overall_acc,
        "adapter_acc": adapter_acc.to_dict(),  # Full dict still returned for use
        "graph_utilization": graph_utilization,
        "alignment_acc": alignment_acc,
        "alignment_stats": alignment_stats,
        "adapter_perf": adapter_perf.to_dict(),
        "adapter_selection_acc": adapter_selection_acc,
        "avg_retention": avg_retention,
        "adapter_contributions": adapter_contributions.to_dict()
    }


# Example usage:
results = evaluate_pipeline("data/evaluations/Graph_plus_baseModel/preference_evaluation_without_lora_updates.csv")


### --- Evaluation Metrics ---

| Metric                                | Value                       |
|---------------------------------------|----------------------------:|
| Overall Accuracy                      | 77.0%                       |
| Graph Utilization Rate                | 100.0%                      |
| Preference Alignment Accuracy         | 88.3%                       |
| Alignment Score — Mean                | 51.4%                       |
| Alignment Score — Std                 | 14.5%                       |
| Alignment Score — Min                 | 33.3%                       |
| Alignment Score — Max                 | 100.0%                      |
| Adapter Selection Accuracy (avg)      | 86.3%                       |
| Average Retention Stability           | 28.0%                       |

---

### --- Adapter Contributions Summary ---

| Adapter  | Queries Handled | Accuracy  |
|---------:|----------------:|---------:|
| blended  | 240             | 80.0%    |
| global   | 21              | 33.3%    |
| local    | 13              | 92.3%    |

Best Adapter: **local** (Accuracy: 92.3%)

In [ ]:
results = evaluate_pipeline("data/evaluations/complete_adaptersPlusBaseModel/combined_llm_results_with_meta.csv")

### --- Evaluation Metrics ---

| Metric                                | Value       |
|---------------------------------------|------------:|
| Overall Accuracy                      | 78.1%       |
| Graph Utilization Rate                | 100.0%      |
| Preference Alignment Accuracy         | 88.9%       |
| Alignment Score — Mean                | 51.3%       |
| Alignment Score — Std                 | 14.3%       |
| Alignment Score — Min                 | 33.3%       |
| Alignment Score — Max                 | 100.0%      |
| Adapter Selection Accuracy (avg)      | 78.1%       |
| Average Retention Stability           | 28.0%       | 

---

### --- Adapter Contributions Summary ---

| Adapter   | Queries Handled | Accuracy  |
|----------:|----------------:|---------:|
| combined  | 270             | 78.1%    |

Best Adapter: **combined** (Accuracy: 78.1%)

# Only Graph + Base Model no Lora updates

### --- Evaluation Metrics ---
| Metric                           | Value  |
|----------------------------------|--------|
| Overall Accuracy                 | 57.6%  |
| Graph Utilization Rate           | 100.0% |
| Preference Alignment Accuracy    | 85.6%  |
| Alignment Score — Mean           | 48.0%  |
| Alignment Score — Std            | 13.4%  |
| Alignment Score — Min            | 33.3%  |
| Alignment Score — Max            | 100.0% |
| Adapter Selection Accuracy (avg) | 58.0%  |
| Average Retention Stability      | 8.0%   |

### --- Adapter Contributions Summary ---
Best Adapter: local (Accuracy: 74.1%)

| Adapter | Queries Handled | Accuracy |
|--------:|----------------:|---------:|
| blended | 229             | 59.0%    |
| global  | 22              | 22.7%    |
| local   | 27              | 74

# Plotting results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data for plots
methods = ["Baseline", "FedAvg", "Vanilla", "Combined"]
means = [0.40, 0.43, 0.40, 0.41]
stds  = [0.04, 0.06, 0.06, 0.04]

settings = ["Base + Router", "Global + Local Combined"]
overall_acc = [0.77, 0.781]
pref_acc    = [0.883, 0.889]
sel_acc     = [0.863, 0.781]
x = np.arange(len(settings))
width = 0.25

per_fold = {
    "FedAvg": [0.453125, 0.362069, 0.482759, 0.357143, 0.482143],
    "Default": [0.406780, 0.393443, 0.383333, 0.333333, 0.467742],
    "Vanilla": [0.355932, 0.328125, 0.393443, 0.388889, 0.509091],
    "Combined": [0.377358, 0.350000, 0.474576, 0.428571, 0.413793]
}

# Create multi-panel figure
fig, axes = plt.subplots(1, 3, figsize=(12, 3))  # 1 row, 3 columns

# ---- Panel A: Aggregation Strategies ----
axes[0].bar(methods, means, yerr=stds, capsize=6, 
            color=["#4c72b0","#55a868","#c44e52","#8172b2"])
axes[0].set_ylabel("Accuracy", fontsize=10)
axes[0].set_ylim(0, 0.55)
axes[0].set_title("(a) Aggregation Strategies", fontsize=10, weight="bold")
axes[0].grid(axis="y", linestyle="--", alpha=0.7)

# ---- Panel B: Pipeline Results ----
axes[1].bar(x - width, overall_acc, width, label="Overall", color="#4c72b0")
axes[1].bar(x, pref_acc, width, label="Pref Align", color="#55a868")
axes[1].bar(x + width, sel_acc, width, label="Selection", color="#c44e52")
axes[1].set_title("(b) Pipeline Performance", fontsize=10, weight="bold")
axes[1].set_xticks(x)
axes[1].set_xticklabels(settings, rotation=15, fontsize=8)
axes[1].set_ylim(0,1)
axes[1].grid(axis="y", linestyle="--", alpha=0.7)
axes[1].legend(fontsize=7)

# ---- Panel C: Per-Fold Distribution ----
axes[2].boxplot(per_fold.values(), labels=per_fold.keys(), patch_artist=True,
                boxprops=dict(facecolor="#4c72b0", alpha=0.5))
axes[2].set_ylabel("Accuracy", fontsize=10)
axes[2].set_title("(c) Per-Fold Accuracy", fontsize=10, weight="bold")
axes[2].grid(axis="y", linestyle="--", alpha=0.7)

# Save with no extra spacing
plt.tight_layout()
plt.savefig("multi_panel_results.pdf", bbox_inches="tight", pad_inches=0)
plt.close()


In [ ]:
import pandas as pd

def duplicate_counts_by_column(df, column="preference"):
    # Rows where the column value appears more than once
    dup_counts = (
        df[df.duplicated(column, keep=False)]
        .groupby(column, dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )
    return dup_counts

if __name__ == "__main__":
    CSV_PATH = "data/preference_evaluation_results.csv"  # change as needed
    COLUMN = "preference"  # change as needed
    CSV_OUT = "data/preference_evaluation_without_lora_updates.csv"

    df = pd.read_csv(CSV_PATH)
    if COLUMN not in df.columns:
        raise SystemExit(f"Column not found: {COLUMN}")

    dup_df = duplicate_counts_by_column(df, COLUMN)

    if dup_df.empty:
        print(f"No duplicates found for column: {COLUMN}")
    else:
        # Total duplicate rows beyond first occurrences
        extra_dups = int(dup_df["count"].sum() - len(dup_df))
        print(dup_df.to_string(index=False))
        print(f"\nDistinct duplicated values: {len(dup_df)}")
        print(f"Total duplicate rows (beyond first occurrences): {extra_dups}")

    before = len(df)
    df_no_dups = df.drop_duplicates(subset=[COLUMN], keep="first").reset_index(drop=True)
    df_no_dups.to_csv(CSV_OUT, index=False)
    print(f"Saved {len(df_no_dups)} rows to {CSV_OUT} (removed {before - len(df_no_dups)} duplicates)")

In [ ]:
import os
import time
import uuid
import re
import math
import pandas as pd
import numpy as np
import nltk
import torch
from typing import List, Tuple
from tqdm import tqdm

from sklearn.metrics import precision_score, recall_score, f1_score
from transformers import BertTokenizer, BertModel

# Pinecone: adapt to the style you already used
from pinecone import Pinecone, ServerlessSpec

# ===========================
# Setup
# ===========================
nltk.download("stopwords", quiet=True)
STOPWORDS = set(nltk.corpus.stopwords.words("english"))
KEEP_WORDS = {"prefer", "dislike", "like", "love", "hate", "enjoy", "avoid"}
STOPWORDS = STOPWORDS - KEEP_WORDS

# BERT tokenizer/model for embeddings
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")
bert_model.eval()

# ===========================
# Pinecone Init (configure)
# ===========================
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
if not PINECONE_API_KEY:
    raise RuntimeError("PINECONE_API_KEY not found in environment")

INDEX_NAME = "preference-prompts"  # ensure this index exists or create it separately

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("preference-prompts")

# ===========================
# Helpers
# ===========================
def preprocess(text: str) -> List[str]:
    text = (text or "").lower()
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = text.split()
    return [t for t in tokens if t not in STOPWORDS]

def keyword_alignment(preferences: List[str], query: str, threshold: float = 0.3
                     ) -> Tuple[List[Tuple[str, float, List[str]]], List[Tuple[str, float, List[str]]]]:
    query_kw = set(preprocess(query))
    if not query_kw:
        return [], []
    
    results = []
    for pref in preferences:
        pref_kw = set(preprocess(pref))
        if not pref_kw:
            continue
        common = query_kw.intersection(pref_kw)
        score = len(common) / min(len(query_kw), len(pref_kw))
        if score >= threshold:
            results.append((pref, score, list(common)))
    
    if not results:
        return [], []
    
    max_score = max(r[1] for r in results)
    best = [r for r in results if r[1] == max_score]
    
    return results, best

def embed_text(text: str) -> np.ndarray:
    """Return 1D numpy vector embedding using BERT pooling (CLS token)."""
    if text is None:
        text = ""
    inputs = bert_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    emb = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
    return emb

def chunked_upsert(index_obj, vectors: List[dict], chunk_size: int = 100):
    """Upsert vectors in chunks to avoid oversized requests."""
    for i in range(0, len(vectors), chunk_size):
        chunk = vectors[i:i+chunk_size]
        index_obj.upsert(vectors=chunk)

def safe_describe_count(index_obj) -> int:
    """Try to get total_vector_count from describe_index_stats if available."""
    try:
        stats = index_obj.describe_index_stats()
        # depending on SDK, stats structure differs; try a couple of keys
        if hasattr(stats, "total_vector_count"):
            return int(stats.total_vector_count)
        if isinstance(stats, dict):
            # pinecone serverless style: {'total_vector_count': n, ...}
            return int(stats.get("total_vector_count", 0))
    except Exception:
        pass
    # fallback: unknown / 0
    return 0

# ===========================
# Main Pipeline (per-user evaluation)
# ===========================
def evaluate_preferences_pinecone(csv_path: str,
                                  detailed_output_path: str,
                                  summary_output_path: str,
                                  top_k: int = 5,
                                  threshold: float = 0.3):
    df = pd.read_csv(csv_path)
    df["userID"] = "user10"
    if "userID" not in df.columns:
        raise ValueError("CSV must contain a 'userID' column")

    # Prepare outputs
    detailed_rows = []
    summary_rows = []

    # Iterate users one-by-one: insert only that user's preferences, evaluate, then delete
    unique_users = df["userID"].unique().tolist()
    print(f"Found {len(unique_users)} unique users.")

    for user_id in unique_users:
        print("\n" + "="*40)
        print(f"Processing user: {user_id}")
        print("="*40)

        # --- Clear index before inserting this user's data ---
        try:
            before = safe_describe_count(index)
            print(f"Entries before deletion: {before}")
            index.delete(delete_all=True)  # may raise depending on SDK/version
            time.sleep(0.5)
            after = safe_describe_count(index)
            print(f"Entries after deletion: {after}")
        except Exception as e:
            print(f"[Warning] Failed to fully clear index: {e}")

        # --- Upsert this user's preference_prompts ---
        user_rows = df[df["userID"] == user_id].reset_index(drop=True)
        vectors = []
        for i, row in user_rows.iterrows():
            pref_text = row.get("preference_prompt", "")
            pref_id = f"{user_id}-{uuid.uuid4()}"
            emb = embed_text(pref_text)
            vectors.append({"id": pref_id, "values": emb.tolist(),
                            "metadata": {"userID": user_id, "text": pref_text}})
        if vectors:
            chunked_upsert(index, vectors, chunk_size=100)
            print(f"Upserted {len(vectors)} preference vectors for user {user_id}")
        else:
            print(f"No preference vectors found for user {user_id}; skipping upsert")

        # Verify insertion
        time.sleep(0.5)
        inserted_count = safe_describe_count(index)
        print(f"Entries after upsert (approx): {inserted_count}")

        # --- Query the index for this user's queries only ---
        total, correct = 0, 0
        y_true, y_pred = [], []
        coverage_hits = 0
        all_scores = []
        latencies = []

        # We'll iterate only the rows for that user (their queries)
        for _, row in user_rows.iterrows():
            query_text = row.get("question_prompt", "")
            gold_answer = row.get("answer", "")

            query_emb = embed_text(query_text)

            # measure retrieval latency
            start = time.time()
            try:
                res = index.query(vector=query_emb.tolist(), top_k=top_k,
                                  filter={"userID": {"$eq": user_id}}, include_metadata=True)
            except Exception as e:
                # If query fails, record empty result and continue
                print(f"[Warning] query failed for user {user_id}: {e}")
                res = {"matches": []}
            latency = time.time() - start
            latencies.append(latency)

            matches = res.get("matches", []) if isinstance(res, dict) else (res.matches if hasattr(res, "matches") else [])
            preferences = []
            for m in matches:
                md = m.get("metadata", {}) if isinstance(m, dict) else getattr(m, "metadata", {})
                text = md.get("text") if isinstance(md, dict) else None
                if text:
                    preferences.append(text)

            aligned_prefs, best_prefs = keyword_alignment(preferences, query_text, threshold=threshold)

            for _, score, _ in aligned_prefs:
                all_scores.append(score)

            if preferences:
                total += 1
                if aligned_prefs:
                    correct += 1

            y_true.append(1 if (isinstance(gold_answer, str) and gold_answer.strip()) else 0)
            y_pred.append(1 if aligned_prefs else 0)

            if preferences and isinstance(gold_answer, str) and gold_answer.strip():
                if any(gold_answer.lower() in pref.lower() for pref in preferences):
                    coverage_hits += 1

            detailed_rows.append({
                "userID": user_id,
                "query_prompt": query_text,
                "gold_answer": gold_answer,
                "retrieved_prefs": preferences,
                "aligned_prefs": aligned_prefs,
                "best_pref": best_prefs,
                "latency": latency
            })

        # --- Compute per-user metrics ---
        accuracy = (correct / total) if total > 0 else 0.0
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        coverage = (coverage_hits / total) if total > 0 else 0.0

        alignment_stats = {
            "align_mean": float(np.mean(all_scores)) if all_scores else 0.0,
            "align_std": float(np.std(all_scores)) if all_scores else 0.0,
            "align_min": float(np.min(all_scores)) if all_scores else 0.0,
            "align_max": float(np.max(all_scores)) if all_scores else 0.0,
            "num_aligned_scores": len(all_scores)
        }

        avg_latency = float(np.mean(latencies)) if latencies else 0.0
        p95_latency = float(np.percentile(latencies, 95)) if latencies else 0.0

        summary = {
            "userID": user_id,
            "num_queries": len(user_rows),
            "num_prefs_upserted": len(vectors),
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "coverage_at_k": coverage,
            "avg_latency_sec": avg_latency,
            "p95_latency_sec": p95_latency,
            **alignment_stats
        }
        summary_rows.append(summary)

        # Print per-user metrics
        print(f"\nMetrics for user {user_id}:")
        print(f"  Queries: {len(user_rows)}, Prefs upserted: {len(vectors)}")
        print(f"  Accuracy: {accuracy:.3f} ({correct}/{total})")
        print(f"  Precision: {precision:.3f}")
        print(f"  Recall:    {recall:.3f}")
        print(f"  F1 Score:  {f1:.3f}")
        print(f"  Coverage@{top_k}: {coverage:.3f} ({coverage_hits}/{total})")
        if alignment_stats["num_aligned_scores"] > 0:
            print(f"  Alignment scores mean/std/min/max: "
                  f"{alignment_stats['align_mean']:.3f}/{alignment_stats['align_std']:.3f}/"
                  f"{alignment_stats['align_min']:.3f}/{alignment_stats['align_max']:.3f}")
        print(f"  Avg latency: {avg_latency:.4f} sec, P95 latency: {p95_latency:.4f} sec")

        # --- Delete this user's vectors before next user (to ensure clean slate) ---
        try:
            index.delete(delete_all=True)
            time.sleep(0.3)
            after_delete = safe_describe_count(index)
            print(f"Index cleared after evaluating user {user_id} (approx count: {after_delete})")
        except Exception as e:
            print(f"[Warning] Could not clear index after user {user_id}: {e}")

    # --- Save outputs ---
    detailed_df = pd.DataFrame(detailed_rows)
    summary_df = pd.DataFrame(summary_rows)

    detailed_df.to_csv(detailed_output_path, index=False)
    summary_df.to_csv(summary_output_path, index=False)

    print(f"\nDetailed per-query results written to: {detailed_output_path}")
    print(f"Per-user summary metrics written to: {summary_output_path}")

# Example usage:
evaluate_preferences_pinecone(
    csv_path="data/preference_cleaned.csv",
    detailed_output_path="./pinecone_detailed_results.csv",
    summary_output_path="./pinecone_user_summary.csv",
    top_k=5,
    threshold=0.3
)


In [ ]:
import pandas as pd

# Load your CSV file
df = pd.read_csv("./pinecone_user_summary.csv")

# Drop non-numeric columns
numeric_df = df.drop(columns=["userID"])

# Compute averages
avg_metrics = numeric_df.mean(numeric_only=True)

# Save to CSV (optional)
avg_metrics.to_csv("avg_metrics.csv", header=["average"])

# Print nicely
print("=== Average Metrics Across All Users ===")
for col, val in avg_metrics.items():
    print(f"{col}: {val:.4f}")


#### Per user-pinecone

| Metric              | Average   |
|----------------------|-----------|
| num_queries          | 79.6000   |
| num_prefs_upserted   | 79.6000   |
| accuracy             | 0.8082    |
| precision            | 1.0000    |
| recall               | 0.6763    |
| f1                   | 0.7950    |
| coverage_at_k        | 0.3254    |
| avg_latency_sec      | 0.7912    |
| p95_latency_sec      | 1.3765    |
| align_mean           | 0.4374    |
| align_std            | 0.1234    |
| align_min            | 0.3333    |
| align_max            | 0.8014    |
| num_aligned_scores   | 93.4000   |
